In [2]:
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

model = "claude-sonnet-4-5"

In [3]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [14]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to test a prompt that helps users write AWS-related code in one of three formats: Python, JSON configuration, or a regular expression.

Generate an array of exactly 3 JSON objects. Each object must have a "task" property: a short description of something a user might ask for that needs a Python function, a JSON configuration object, or a regular expression.

Keep each task simple: a single function, a single JSON object, or a single regex, with very little code required.

Example format:
[
  {"task": "Description of task"},
  {"task": "Description of task"}
]
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
dataset = generate_dataset()
print(dataset)

C:\Users\andres.lazarte.it\AppData\Local\Temp\ipykernel_26844\313891637.py:20: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


[{'task': 'Write a Python function that takes an S3 bucket name and object key, then returns the full S3 URI in the format s3://bucket/key'}, {'task': 'Create a JSON configuration for an AWS Lambda function with 512MB memory, 30 second timeout, and Python 3.11 runtime'}, {'task': 'Write a regular expression that validates an AWS IAM role ARN format like arn:aws:iam::123456789012:role/RoleName'}]


In [16]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [17]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

print(dataset)

[{'task': 'Write a Python function that takes an S3 bucket name and object key, then returns the full S3 URI in the format s3://bucket/key'}, {'task': 'Create a JSON configuration for an AWS Lambda function with 512MB memory, 30 second timeout, and Python 3.11 runtime'}, {'task': 'Write a regular expression that validates an AWS IAM role ARN format like arn:aws:iam::123456789012:role/RoleName'}]


In [18]:
def run_prompt(test_case):
    prompt = f"Please solve the following task:\n\n{test_case['task']}"

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [19]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    # Placeholder: real grading comes in the next lesson
    score = 10

    return {"output": output, "test_case": test_case, "score": score}

In [20]:
def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [21]:
results = run_eval(dataset)
print(json.dumps(results, indent=2))

C:\Users\andres.lazarte.it\AppData\Local\Temp\ipykernel_26844\313891637.py:20: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(**params)


[
  {
    "output": "Here's a Python function that creates an S3 URI from a bucket name and object key:\n\n```python\ndef create_s3_uri(bucket_name: str, object_key: str) -> str:\n    \"\"\"\n    Create an S3 URI from bucket name and object key.\n    \n    Args:\n        bucket_name: The name of the S3 bucket\n        object_key: The key (path) of the object in the bucket\n    \n    Returns:\n        A string in the format s3://bucket/key\n    \n    Example:\n        >>> create_s3_uri(\"my-bucket\", \"path/to/file.txt\")\n        's3://my-bucket/path/to/file.txt'\n    \"\"\"\n    # Remove leading slash from key if present\n    object_key = object_key.lstrip('/')\n    \n    return f\"s3://{bucket_name}/{object_key}\"\n```\n\n**Usage examples:**\n\n```python\n# Basic usage\nuri = create_s3_uri(\"my-bucket\", \"data/file.csv\")\nprint(uri)  # Output: s3://my-bucket/data/file.csv\n\n# With leading slash in key (will be handled)\nuri = create_s3_uri(\"my-bucket\", \"/data/file.csv\")\nprint

In [22]:
for r in results:
    print("TASK:", r["test_case"]["task"])
    print("SCORE:", r["score"])
    print(r["output"])
    print("-" * 40)

TASK: Write a Python function that takes an S3 bucket name and object key, then returns the full S3 URI in the format s3://bucket/key
SCORE: 10
Here's a Python function that creates an S3 URI from a bucket name and object key:

```python
def create_s3_uri(bucket_name: str, object_key: str) -> str:
    """
    Create an S3 URI from bucket name and object key.
    
    Args:
        bucket_name: The name of the S3 bucket
        object_key: The key (path) of the object in the bucket
    
    Returns:
        A string in the format s3://bucket/key
    
    Example:
        >>> create_s3_uri("my-bucket", "path/to/file.txt")
        's3://my-bucket/path/to/file.txt'
    """
    # Remove leading slash from key if present
    object_key = object_key.lstrip('/')
    
    return f"s3://{bucket_name}/{object_key}"
```

**Usage examples:**

```python
# Basic usage
uri = create_s3_uri("my-bucket", "data/file.csv")
print(uri)  # Output: s3://my-bucket/data/file.csv

# With leading slash in key (wil